# Gold Layer — Clinical Reports & KPIs
**Medallion Healthcare Analytics Platform**  
Celebal Excellence Internship 2025

> The Gold layer aggregates cleaned Silver data into 5 production-ready clinical reports and continuously scores patients for deterioration risk using the trained ML model.

## Gold Layer Outputs
| # | Report | Rows | Consumers |
|---|--------|------|-----------|
| 01 | Hourly Vitals Summary | ~3,131 | Clinicians, Ward Nurses |
| 02 | Deterioration Risk Scores | 50 | ICU Teams, Duty Doctors |
| 03 | Alert Response Times | 24 | Clinical Governance |
| 04 | ICU Capacity Utilisation | 288 | Hospital Operations |
| 05 | High-Risk Patient Detection | varies | All Medical Staff |

## 1. Setup

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
from config.settings import SILVER_VITALS_PATH, GOLD_DIR, RISK_THRESHOLD
silver = pd.read_parquet(SILVER_VITALS_PATH)
print(f'Silver rows: {len(silver):,}  |  Patients: {silver["patient_id"].nunique()}')
silver.head(3)

## 2. Report 01 — Hourly Vitals Summary

In [ ]:
silver['hour_bucket'] = pd.to_datetime(silver['timestamp']).dt.floor('h')

hourly = silver.groupby(['patient_id','hour_bucket']).agg(
    avg_heart_rate      =('heart_rate',       'mean'),
    avg_spo2            =('spo2_pct',          'mean'),
    avg_respiratory_rate=('respiratory_rate',  'mean'),
    avg_systolic_bp     =('systolic_bp',       'mean'),
    avg_temperature_c   =('temperature_c',     'mean'),
    avg_sepsis_risk     =('sepsis_risk_score', 'mean'),
    vital_flag_count    =('vital_flag',        'sum'),
    total_readings      =('heart_rate',        'count'),
).reset_index().round(2)

print(f'Report 01 rows: {len(hourly):,}')
hourly.head(5)

## 3. Report 02 — ML Risk Score per Patient

In [ ]:
import joblib, json
from ml.train_model import engineer_features, get_feature_columns

model  = joblib.load('ml/model/deterioration_rf_model.joblib')
scaler = joblib.load('ml/model/feature_scaler.joblib')

with open('ml/model/feature_columns.json') as f:
    feat_cols = json.load(f)

# Engineer same features used during training
df_eng = engineer_features(silver)
X = df_eng.reindex(columns=feat_cols, fill_value=0).fillna(0).values
X_scaled = scaler.transform(X)
silver['risk_score'] = model.predict_proba(X_scaled)[:,1]

# Latest score per patient
risk_report = silver.sort_values('timestamp').groupby('patient_id').last().reset_index()
risk_report['risk_flag'] = (risk_report['risk_score'] >= RISK_THRESHOLD).astype(int)
print(f'Report 02 rows: {len(risk_report)}')
print(f'High-risk (>={RISK_THRESHOLD*100:.0f}%): {risk_report["risk_flag"].sum()}')
risk_report[['patient_id','risk_score','risk_flag']].sort_values('risk_score',ascending=False).head(10)

## 4. Report 03 — Alert Response Times

In [ ]:
from config.settings import BRONZE_DIR
alerts = pd.read_parquet(os.path.join(BRONZE_DIR,'alert_events_raw.parquet'))
intv   = pd.read_parquet(os.path.join(BRONZE_DIR,'interventions_raw.parquet'))
merged = alerts.merge(intv[['alert_id','response_time_min','action']], on='alert_id', how='left')
sla_map = {'Critical':5,'High':10,'Medium':20,'Low':45}
merged['sla_minutes'] = merged['severity'].map(sla_map)
merged['sla_met']     = (merged['response_time_min'] <= merged['sla_minutes']).astype(int)
summary = merged.groupby('severity').agg(
    total_alerts=('alert_id','count'),
    avg_response_min=('response_time_min','mean'),
    sla_met_pct=('sla_met','mean')
).reset_index()
summary['sla_met_pct'] = (summary['sla_met_pct']*100).round(1)
summary['avg_response_min'] = summary['avg_response_min'].round(1)
print('Report 03:')
summary

## 5. Report 04 — ICU Capacity

In [ ]:
icu = pd.read_parquet(os.path.join(BRONZE_DIR,'icu_capacity_raw.parquet'))
icu['utilisation_pct'] = (icu['occupied_beds']/icu['total_beds']*100).round(1)
icu_summary = icu.groupby('unit_id').agg(
    total_beds=('total_beds','first'),
    avg_occupied=('occupied_beds','mean'),
    avg_util_pct=('utilisation_pct','mean'),
    peak_util_pct=('utilisation_pct','max')
).reset_index().round(1)
print('Report 04 — ICU Capacity:')
icu_summary

## 6. Report 05 — High-Risk Patients

In [ ]:
high_risk = risk_report[risk_report['risk_score'] >= RISK_THRESHOLD].copy()
high_risk['recommended_action'] = high_risk['risk_score'].apply(
    lambda s: 'Immediate ICU Review' if s>=0.75 else
              'Urgent Ward Review'   if s>=0.55 else
              'Increase Monitoring Frequency'
)
print(f'High-risk patients: {len(high_risk)}')
print('\nRecommended actions:')
print(high_risk['recommended_action'].value_counts())
high_risk[['patient_id','risk_score','recommended_action']].sort_values('risk_score',ascending=False)

## 7. Run Full Gold Layer via Pipeline

In [ ]:
from pipeline.gold_layer import GoldLayer
gl = GoldLayer()
reports = gl.run()
print('\nAll Gold reports generated:')
for name, df in reports.items():
    print(f'  {name:<35} {len(df):>6,} rows')

## Databricks SQL Equivalent
```sql
-- Gold Layer: Hourly Vitals Summary
CREATE OR REPLACE TABLE gold.hourly_vitals_summary AS
SELECT
    patient_id,
    DATE_TRUNC('hour', timestamp) AS hour_bucket,
    AVG(heart_rate)       AS avg_heart_rate,
    AVG(spo2_pct)         AS avg_spo2,
    AVG(respiratory_rate) AS avg_respiratory_rate,
    AVG(systolic_bp)      AS avg_systolic_bp,
    AVG(temperature_c)    AS avg_temperature_c,
    SUM(vital_flag)       AS vital_flag_count,
    COUNT(*)              AS total_readings
FROM silver.vitals_clean
GROUP BY patient_id, DATE_TRUNC('hour', timestamp);
```